In [0]:
# =====================================================================
# CELL 1: WORKFLOW AS CODE -> create (or update) the daily pipeline job
# =====================================================================
# Each TASK runs one notebook; depends_on defines the order (a DAG).
# Serverless compute: no cluster config needed.
# Schedule is PAUSED: we trigger runs manually for the demo (saves free quota).
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w    = WorkspaceClient()
user = spark.sql("SELECT current_user()").first()[0]
BASE = f"/Users/{user}/procintel"           # the folder with our notebooks
NAME = "procintel_daily_pipeline"

def task(key, notebook, depends=()):
    return jobs.Task(
        task_key=key,
        notebook_task=jobs.NotebookTask(notebook_path=f"{BASE}/{notebook}"),
        depends_on=[jobs.TaskDependency(task_key=d) for d in depends],
        timeout_seconds=3600,                # kill a stuck task after 1 hour
        max_retries=1)                       # retry once on transient failures (APIs flake)

settings = dict(
    name=NAME,
    tasks=[
        task("ingest_bronze", "02_ingest_bronze"),
        task("bronze_tables", "03_bronze_tables", ["ingest_bronze"]),
        task("silver",        "04_silver",        ["bronze_tables"]),
        task("gold",          "05_gold",          ["silver"]),
        task("train_model",   "06_train_model",   ["gold"]),           # these two run
        task("documents",     "07_documents",     ["gold"]),           # in PARALLEL
        task("monitoring",    "09_monitoring",    ["train_model", "documents"]),
    ],
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 6 * * ?",  # every day 06:00
                               timezone_id="America/New_York",
                               pause_status=jobs.PauseStatus.PAUSED),
    email_notifications=jobs.JobEmailNotifications(on_failure=[user]),  # alert me on failure
    max_concurrent_runs=1,                   # never two pipeline runs at once
    tags={"project": "procintel"})

existing = list(w.jobs.list(name=NAME))
if existing:
    job_id = existing[0].job_id
    w.jobs.reset(job_id=job_id, new_settings=jobs.JobSettings(**settings))   # update in place
else:
    job_id = w.jobs.create(**settings).job_id
print("job id:", job_id)

run = w.jobs.run_now(job_id=job_id)          # start one run now (don't wait)
print("run started:", run.run_id, "-> watch it under Jobs & Pipelines")